# Random Qwen2.5-7B — Codenames Duet layer extraction

Architecture-only control for Qwen2.5-7B-Instruct: the same model class is instantiated with randomly drawn weights (from the model's own initializer). The package and notebook orchestration are otherwise identical to the trained-Qwen notebook (`02_qwen.ipynb`).

**Pre-flight is required.** Before running the main extraction loop, a small diagnostic forward pass on 5 boards verifies the random network produces numerically stable hidden states (no NaN/Inf, no runaway norm growth, plausible L0 anisotropy). Do not skip preflight: if it fails, the main loop output is meaningless.

**Two ways to run.** Cells 1–3 are the shared Colab setup (clone, install, mount). After setup, choose **Path A** (CLI batch run) *or* **Path B** (cell-by-cell), not both:

- **Path A — Full CLI invocation.** Two cells: pre-flight first, then the full run.
- **Path B — Cell-by-cell.** Load model → pre-flight diagnostic → load dataset → run extraction → SC1 → … → SC7 → output summary.

## Setup (run cells 1–3 once per session)

In [ ]:
# Cell 1 — Clone or update the package code from GitHub.
import os
REPO_URL = "https://github.com/JoaoPedroFPK/codenames-interpretability.git"
REPO_DIR = "/content/codenames-interpretability"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

In [ ]:
# Cell 2 — Install the package in editable mode.
!pip install -q -e {REPO_DIR}

In [ ]:
# Cell 3 — Autoreload and mount Drive.
%load_ext autoreload
%autoreload 2

from google.colab import drive
drive.mount("/content/drive")

## Path A — Full CLI invocation (batch run)

Run the **preflight** cell first. Only proceed to the **run** cell if preflight reports `ALL HARD CHECKS PASSED`.

Skip both cells if you intend to run Path B instead.

In [ ]:
# Path A preflight
!codenames-experiment preflight \
    --model qwen_random \
    --dataset /content/drive/MyDrive/TCC/clue_generation.csv

In [ ]:
# Path A run
!codenames-experiment run \
    --model qwen_random \
    --dataset /content/drive/MyDrive/TCC/clue_generation.csv \
    --output-dir /content/drive/MyDrive/TCC/random_qwen_outputs

## Path B — Cell-by-cell (interactive verification)

Run the cells below one at a time. The **pre-flight diagnostic** cell (Cell 6) must report all hard checks PASS before you proceed to Cell 7 (Run extraction).

Skip these cells if you already ran Path A.

In [ ]:
# Cell 4 — Imports and config (no generation for random-init causal model).
from codenames_interpretability.contract import CONTRACT_V1
from codenames_interpretability.data import (
    load_dataset, sample_turns, GIVER_COLS, extract_giver_features,
)
from codenames_interpretability.models.qwen_random import load_qwen_random
from codenames_interpretability.diagnostics import preflight_random_init
from codenames_interpretability.loop import run_extraction
from codenames_interpretability.sanity import (
    sc1_prompt_structure, sc2_span_coverage, sc3_anisotropy,
    sc4_behavioral_accuracy, sc5_layer_margin_curve,
    sc6_positional_confound, sc7_shuffle_decomposition,
)
from codenames_interpretability.persistence import print_output_summary

DATASET_PATH = "/content/drive/MyDrive/TCC/clue_generation.csv"

In [ ]:
# Cell 5 — Load model (random init via accelerate + RMSNorm re-init).
model, tokenizer, meta = load_qwen_random()
print(f"Model loaded: {meta['model_name']}")
print(f"  Layers: {meta['num_layers']}, Hidden dim: {meta['hidden_dim']}")

In [ ]:
# Cell 6 — Pre-flight diagnostic. Do NOT proceed to Cell 7 if this fails.
df = load_dataset(DATASET_PATH)
preflight = preflight_random_init(
    df=df,
    model=model,
    tokenizer=tokenizer,
    device=meta["device"],
    num_layers=meta["num_layers"],
    chat_template_strategy=meta["chat_template_strategy"],
    n_boards=5,
)
assert preflight["all_hard_checks_pass"], (
    "Pre-flight failed; refusing to run main extraction loop. "
    "See the diagnostic output above and follow the fallback instructions."
)

In [ ]:
# Cell 7 — Sample dataset.
df_sample = sample_turns(df, n=CONTRACT_V1.sample_size, seed=CONTRACT_V1.random_seed)
print(f"Sample size: {len(df_sample)} boards")
print(f"First 10 row_ids: {sorted(df_sample['row_id'].tolist())[:10]}")

In [ ]:
# Cell 8 — Run extraction (no generation).
BASE_DIR = f"/content/drive/MyDrive/TCC/{meta['prefix']}_outputs"

results = run_extraction(
    model=model,
    tokenizer=tokenizer,
    df=df_sample,
    base_dir=BASE_DIR,
    prefix=meta["prefix"],
    contract=CONTRACT_V1,
    chat_template_strategy=meta["chat_template_strategy"],
    forward_hidden_states_mode=meta["forward_hidden_states_mode"],
    use_truncation=meta["use_truncation"],
    num_layers=meta["num_layers"],
    hidden_dim=meta["hidden_dim"],
    device=meta["device"],
    has_generation=False,
    generation_fn=None,
)

In [ ]:
# Cell 9 — SC1.
sc1_prompt_structure(df_sample, tokenizer, meta["chat_template_strategy"])

In [ ]:
# Cell 10 — SC2.
sc2_span_coverage(results)

In [ ]:
# Cell 11 — SC3.
sc3_anisotropy(results, num_layers=meta["num_layers"])

In [ ]:
# Cell 12 — SC4.
sc4_behavioral_accuracy(
    results,
    pooling_methods=CONTRACT_V1.pooling_methods,
    has_generation=False,
)

In [ ]:
# Cell 13 — SC5.
sc5_layer_margin_curve(
    results, base_dir=BASE_DIR, prefix=meta["prefix"],
    num_layers=meta["num_layers"], pooling_methods=CONTRACT_V1.pooling_methods,
)

In [ ]:
# Cell 14 — SC6.
sc6_positional_confound(
    results, base_dir=BASE_DIR, prefix=meta["prefix"],
    num_layers=meta["num_layers"],
)

In [ ]:
# Cell 15 — SC7.
sc7_shuffle_decomposition(
    results, base_dir=BASE_DIR, prefix=meta["prefix"],
    num_layers=meta["num_layers"], n_shuffles=CONTRACT_V1.n_shuffles,
)

In [ ]:
# Cell 16 — Output summary.
print_output_summary(
    base_dir=BASE_DIR, prefix=meta["prefix"], contract=CONTRACT_V1,
    has_generation=False,
    pooling_methods=CONTRACT_V1.pooling_methods,
)